# TD4 — Knowledge Base Construction, Alignment & Expansion (Fixed)

Continues directly from **TD1** outputs.  

**Outputs (in `td4/outputs/`):**
- `entity_mapping.csv` — Wikidata QID per entity
- `entity_alignment.ttl` — owl:sameAs statements
- `predicate_alignment.csv`
- `predicate_alignment.ttl` — owl:equivalentProperty statements
- `expanded_kb.ttl` — full expanded Knowledge Base
- `expanded_kb.nt` — same, N-Triples format
- `final_kb_stats.csv`

In [ ]:
# ── 0. Dependencies ────────────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f'Installing {pip_name}…')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])

for p, n in [('rdflib', None), ('SPARQLWrapper', 'SPARQLWrapper'), ('pandas', None), ('requests', None)]:
    ensure(p, n)

import re, time
from collections import Counter, deque
from pathlib import Path

import pandas as pd
import requests
from rdflib import Graph, Literal, Namespace, RDFS, OWL, RDF, URIRef, XSD
from SPARQLWrapper import SPARQLWrapper, JSON

print('Imports OK')

In [ ]:
# ── 1. Paths & namespaces ──────────────────────────────────────────────────
def find_root(start: Path) -> Path:
    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / 'td1').exists() and (p / 'td4').exists():
            return p
    raise FileNotFoundError('Cannot find workspace root')

ROOT = find_root(Path.cwd())
TD1  = ROOT / 'td1'
TD4  = ROOT / 'td4'
OUT  = TD4 / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

# Input
ENTITIES_CSV  = TD1 / 'outputs' / 'extracted_knowledge.csv'
RELATIONS_CSV = TD1 / 'outputs' / 'relation_candidates.csv'

# Output
MAPPING_CSV             = OUT / 'entity_mapping.csv'
ENTITY_ALIGNMENT_TTL    = OUT / 'entity_alignment.ttl'
PREDICATE_ALIGNMENT_CSV = OUT / 'predicate_alignment.csv'
PREDICATE_ALIGNMENT_TTL = OUT / 'predicate_alignment.ttl'
EXPANDED_TTL            = OUT / 'expanded_kb.ttl'
EXPANDED_NT             = OUT / 'expanded_kb.nt'
FINAL_STATS_CSV         = OUT / 'final_kb_stats.csv'

# Namespaces
MYKB = Namespace('http://private.org/energy/')
WDT  = Namespace('http://www.wikidata.org/prop/direct/')
WD   = Namespace('http://www.wikidata.org/entity/')

# Load TD1 outputs
df_entities  = pd.read_csv(ENTITIES_CSV)
df_relations = pd.read_csv(RELATIONS_CSV)

# Normalise column names (TD1 may use 'subject'/'predicate'/'object' or 'source'/'relation'/'target')
col_s = 'subject'   if 'subject'   in df_relations.columns else 'source'
col_p = 'predicate' if 'predicate' in df_relations.columns else 'relation'
col_t = 'object'    if 'object'    in df_relations.columns else 'target'

print(f'ROOT            : {ROOT}')
print(f'Entities CSV    : {ENTITIES_CSV}  exists={ENTITIES_CSV.exists()}')
print(f'Relations CSV   : {RELATIONS_CSV}  exists={RELATIONS_CSV.exists()}')
print(f'Entity rows     : {len(df_entities)}')
print(f'Relation rows   : {len(df_relations)}')
print(f'Relation cols   : {df_relations.columns.tolist()}')

In [ ]:
# ── 2. Build private RDF graph from TD1 triples ────────────────────────────
def slugify(text: str) -> str:
    t = re.sub(r'[^A-Za-z0-9]+', '_', str(text).strip())
    return re.sub(r'_+', '_', t).strip('_') or 'entity'

def entity_uri(label: str) -> URIRef:
    return MYKB[f'entity/{slugify(label)}']

def prop_uri(rel: str) -> URIRef:
    return MYKB[f'prop/{slugify(rel.lower())}']

g = Graph()
g.bind('mykb', MYKB)
g.bind('owl', OWL)
g.bind('rdfs', RDFS)
g.bind('rdf', RDF)
g.bind('wdt', WDT)
g.bind('wd', WD)

# Add entity type triples
entity_col = 'Entity_Name' if 'Entity_Name' in df_entities.columns else 'Entity'
type_col   = 'Type'        if 'Type'        in df_entities.columns else 'label'

for _, row in df_entities.iterrows():
    uri = entity_uri(str(row[entity_col]))
    g.add((uri, RDFS.label, Literal(str(row[entity_col]))))
    g.add((uri, RDF.type, MYKB[f'type/{str(row[type_col]).upper()}']))

# Add relation triples
for _, row in df_relations.iterrows():
    src = str(row.get(col_s, '')).strip()
    rel = str(row.get(col_p, '')).strip()
    tgt = str(row.get(col_t, '')).strip()
    if not (src and rel and tgt):
        continue
    s_uri = entity_uri(src)
    p_uri = prop_uri(rel)
    t_uri = entity_uri(tgt)
    g.add((s_uri, p_uri, t_uri))
    g.add((s_uri, RDFS.label, Literal(src)))
    g.add((t_uri, RDFS.label, Literal(tgt)))
    g.add((p_uri, RDFS.label, Literal(rel)))

print(f'Private KG triples after loading TD1: {len(g)}')

In [ ]:
# ── 3. Entity linking to Wikidata (owl:sameAs) ────────────────────────────
WIKIDATA_SEARCH = 'https://www.wikidata.org/w/api.php'

session = requests.Session()
session.headers.update({'User-Agent': 'TD4-KB-Alignment/1.0 (academic project)'})

def wikidata_search(label: str, limit: int = 3) -> list:
    resp = session.get(WIKIDATA_SEARCH, params={
        'action': 'wbsearchentities', 'format': 'json',
        'language': 'en', 'type': 'item',
        'search': label, 'limit': str(limit),
    }, timeout=20)
    resp.raise_for_status()
    return resp.json().get('search', [])

def confidence_score(private_label: str, wd_label: str) -> float:
    a, b = private_label.lower().strip(), wd_label.lower().strip()
    if a == b:       return 0.97
    if a in b or b in a: return 0.90
    return 0.75

def is_valid_label(label: str) -> bool:
    v = str(label).strip()
    return (2 <= len(v) <= 80
            and bool(re.search(r'[A-Za-z]', v))
            and sum(1 for c in v if not c.isalnum() and not c.isspace()) / max(1, len(v)) <= 0.25)

# Collect unique label candidates from both entity & relation tables
entity_labels    = df_entities[entity_col].astype(str).str.strip().tolist()
relation_labels  = (df_relations[col_s].astype(str).str.strip().tolist()
                   + df_relations[col_t].astype(str).str.strip().tolist())
all_labels       = [l for l in (entity_labels + relation_labels) if is_valid_label(l)]

counts    = Counter(l.lower() for l in all_labels)
canonical = {}
for l in all_labels:
    canonical.setdefault(l.lower(), l)

# Limit to top-N by frequency to avoid rate-limiting
MAX_LINK = 500  # Increase if you have time; each call sleeps 0.3 s
ranked   = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:MAX_LINK]
unique_labels = [canonical[k] for k, _ in ranked]
print(f'Unique labels to link: {len(unique_labels)}')

mapping_rows = []
for idx, label in enumerate(unique_labels, 1):
    private_uri  = entity_uri(label)
    external_uri = ''
    conf         = 0.0

    for attempt in range(1, 4):
        try:
            results = wikidata_search(label)
            if results:
                best = results[0]
                qid  = str(best.get('id', '')).strip()
                if qid:
                    external_uri = f'http://www.wikidata.org/entity/{qid}'
                    conf = confidence_score(label, str(best.get('label', '')))
                    g.add((private_uri, OWL.sameAs, URIRef(external_uri)))
            break
        except Exception:
            time.sleep(1.5 * attempt)

    mapping_rows.append({'private_label': label, 'external_uri': external_uri,
                         'confidence_score': conf})
    if idx % 50 == 0:
        print(f'  Linked {idx}/{len(unique_labels)} …')
    time.sleep(0.3)  # Polite delay

mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(MAPPING_CSV, index=False)

# Save entity alignment graph
g_align = Graph()
g_align.bind('owl', OWL)
for s, p, o in g:
    if p == OWL.sameAs:
        g_align.add((s, p, o))
g_align.serialize(destination=str(ENTITY_ALIGNMENT_TTL), format='turtle')

mapped = int((mapping_df['external_uri'] != '').sum())
print(f'\nLinked {mapped}/{len(unique_labels)} entities to Wikidata')
print(f'Entity mapping CSV : {MAPPING_CSV}')
print(f'Alignment TTL      : {ENTITY_ALIGNMENT_TTL}')
mapping_df.head(20)

In [ ]:
# ── 4. Predicate alignment to Wikidata properties ─────────────────────────
# Map frequent relation verbs to Wikidata direct properties
PREDICATE_MAP = {
    'produce':     'P1056',  # product or material produced
    'generate':    'P1056',
    'use':         'P2283',  # uses
    'locate':      'P276',   # location
    'include':     'P527',   # has part
    'develop':     'P178',   # developer
    'create':      'P170',   # creator
    'build':       'P178',
    'train':       'P2283',
    'join':        'P463',   # member of
    'publish':     'P123',   # publisher
    'contribute':  'P3712',  # has goal
    'convene':     'P664',   # organizer
    'seek':        'P101',   # field of work
    'increase':    'P3362',  # operating income
    'select':      'P361',   # part of
    'establish':   'P571',   # inception
    'fund':        'P8324',  # funded by
    'invest':      'P8324',
    'supply':      'P1056',
    'install':     'P1056',
    'reduce':      'P3362',
}

relation_freq = (df_relations[col_p].astype(str).str.strip().str.lower()
                 .value_counts().head(30))

pred_rows = []
g_pred    = Graph()
g_pred.bind('owl', OWL)
g_pred.bind('mykb', MYKB)
g_pred.bind('wdt', WDT)

for rel, freq in relation_freq.items():
    qprop = PREDICATE_MAP.get(rel)
    if not qprop:
        continue
    priv_prop = prop_uri(rel)
    wd_prop   = WDT[qprop]
    g.add((priv_prop, OWL.equivalentProperty, wd_prop))
    g_pred.add((priv_prop, OWL.equivalentProperty, wd_prop))
    pred_rows.append({'private_relation': rel, 'wikidata_property': f'wdt:{qprop}',
                      'frequency': int(freq)})

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(PREDICATE_ALIGNMENT_CSV, index=False)
g_pred.serialize(destination=str(PREDICATE_ALIGNMENT_TTL), format='turtle')

print(f'Analyzed relations  : {len(relation_freq)}')
print(f'Aligned predicates  : {len(pred_df)}')
print(f'Predicate CSV       : {PREDICATE_ALIGNMENT_CSV}')
pred_df

In [ ]:
# ── 5. KB Expansion via Wikidata SPARQL (1-hop) ────────────────────────────
# Only expand entities with high-confidence alignment
MIN_CONF        = 0.75   # Lowered from 0.80 to get more expansion seeds
TARGET_TRIPLES  = 30_000 # Realistic target for a 500-entity seed
PER_ENTITY_LIMIT= 40
MAX_REQUESTS    = 2000

sparql_ep = SPARQLWrapper('https://query.wikidata.org/sparql')
sparql_ep.setReturnFormat(JSON)
sparql_ep.setTimeout(30)
sparql_ep.addCustomHttpHeader('User-Agent', 'TD4-KB-Expansion/1.0')

def fetch_one_hop(wd_uri: str, limit: int = PER_ENTITY_LIMIT) -> list[tuple]:
    """Fetch (predicate, object) pairs from Wikidata for wd_uri."""
    # Only keep direct WDT properties and Wikidata entity objects to avoid noise
    q = f"""
    SELECT ?p ?o WHERE {{
      <{wd_uri}> ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
      FILTER(STRSTARTS(STR(?o), "http://www.wikidata.org/entity/Q"))
    }} LIMIT {limit}
    """
    sparql_ep.setQuery(q)
    res = sparql_ep.query().convert()
    triples = []
    for b in res.get('results', {}).get('bindings', []):
        p = b.get('p', {}).get('value')
        o = b.get('o', {}).get('value')
        if p and o:
            triples.append((p, o))
    return triples

# Reload mapping in case cell 3 was re-run
if 'mapping_df' not in dir() or mapping_df.empty:
    mapping_df = pd.read_csv(MAPPING_CSV)

eligible  = mapping_df[
    mapping_df['external_uri'].astype(str).str.startswith('http') &
    (mapping_df['confidence_score'] >= MIN_CONF)
].copy()

seed_uris = eligible['external_uri'].astype(str).dropna().tolist()
frontier  = deque(seed_uris)
visited   = set()
req_count = 0

print(f'Eligible seed entities : {len(seed_uris)}')
print(f'Starting triples       : {len(g)}')
print(f'Target triples         : {TARGET_TRIPLES}')

while len(g) < TARGET_TRIPLES and frontier and req_count < MAX_REQUESTS:
    wd_uri = frontier.popleft()
    if wd_uri in visited:
        continue
    visited.add(wd_uri)

    ok = False
    for attempt in range(1, 4):
        try:
            neighbors = fetch_one_hop(wd_uri)
            ok = True
            break
        except Exception as exc:
            time.sleep(2 * attempt)

    req_count += 1

    if ok:
        for p_str, o_str in neighbors:
            g.add((URIRef(wd_uri), URIRef(p_str), URIRef(o_str)))
            if o_str not in visited:
                frontier.append(o_str)

    if req_count % 20 == 0:
        print(f'req={req_count:4d} | triples={len(g):6d} | frontier={len(frontier):5d}')

    time.sleep(1.0)  # Polite crawl delay for Wikidata

print(f'\nExpansion complete.')
print(f'SPARQL requests : {req_count}')
print(f'Final triples   : {len(g)}')

In [ ]:
# ── 6. Export & stats ─────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

g.serialize(destination=str(EXPANDED_TTL), format='turtle')
g.serialize(destination=str(EXPANDED_NT),  format='nt', encoding='utf-8')

# Count entities and distinct relation types
entities  = set()
relations = set()
for s, p, o in g:
    if isinstance(s, URIRef):
        entities.add(str(s))
    if isinstance(o, URIRef):
        entities.add(str(o))
    relations.add(str(p))

stats_df = pd.DataFrame([{
    'total_triplets':  len(g),
    'total_entities':  len(entities),
    'total_relations': len(relations),
}])
stats_df.to_csv(FINAL_STATS_CSV, index=False)

print('=' * 50)
print('KB FINAL STATISTICS')
print('=' * 50)
print(f'Total triplets    : {len(g):,}')
print(f'Total entities    : {len(entities):,}')
print(f'Total relations   : {len(relations):,}')
print()
print('Exported files:')
for f in [EXPANDED_TTL, EXPANDED_NT, FINAL_STATS_CSV,
           ENTITY_ALIGNMENT_TTL, PREDICATE_ALIGNMENT_TTL,
           MAPPING_CSV, PREDICATE_ALIGNMENT_CSV]:
    sz = Path(f).stat().st_size if Path(f).exists() else 0
    print(f'  {f.name}  ({sz:,} bytes)')

stats_df

In [ ]:
# ── 7. Quick SPARQL sanity check on the exported graph ──────────────────────
from rdflib import Graph as RGraph

g_check = RGraph()
g_check.parse(str(EXPANDED_TTL), format='turtle')
print(f'Re-loaded triples: {len(g_check)}')

# Q: Which entities have the most outgoing relations?
qr = """
SELECT ?s (COUNT(?p) AS ?cnt)
WHERE { ?s ?p ?o . }
GROUP BY ?s
ORDER BY DESC(?cnt)
LIMIT 10
"""
rows = list(g_check.query(qr))
print('\nTop-10 most connected entities:')
print(f'{"Entity":<70}  Triples')
print('-' * 78)
for r in rows:
    print(f'{str(r.s):<70}  {int(r.cnt)}')

---
## TD4 — Notes for Report

### Entity Alignment Method
- Each entity label from TD1 is sent to the **Wikidata Search API** (`wbsearchentities`).  
- The first hit is used as the candidate; an `owl:sameAs` triple is generated.  
- **Confidence** score: 0.97 (exact match), 0.90 (substring), 0.75 (fuzzy).

### Predicate Alignment Method
- The most frequent relation verbs from `relation_candidates.csv` are matched manually to **Wikidata direct properties** (P-numbers) using a curated mapping table.
- An `owl:equivalentProperty` triple links each private predicate to the Wikidata property.

### Expansion Strategy
- Starting from high-confidence (`≥ 0.75`) aligned URIs as seeds, the pipeline does a **BFS breadth-first traversal** of the Wikidata knowledge graph via SPARQL.
- Only `wdt:` (Wikidata direct properties) with **Wikidata entity objects** are kept to stay in-domain.
- Rate limiting: 1 second between requests; max 2,000 SPARQL calls per run.